
# Experiment 9: Perceptron vs Multilayer Perceptron (A/B Experiment)

**Objective:** Implement and compare a Single-Layer Perceptron (PLA) and a Multilayer Perceptron (MLP), with lightweight hyperparameter tuning.

> The supplied lab sheet specifies the English Handwritten Characters Dataset (3,410 images, 62 classes), preprocessing by resize/flatten/normalize, PLA from scratch, MLP tuning, and evaluation using accuracy, precision, recall, F1-score, confusion matrix, ROC curves and convergence curves.



## Computational choice

The original dataset is relatively small, but 62-class image classification can still make repeated neural-network training slow on a lab machine. Therefore, this notebook keeps tuning intentionally small:

- **2 optimizers:** SGD and Adam
- **2 learning rates:** 0.001 and 0.01
- **1 hidden-layer architecture:** 64 neurons
- **1 activation:** ReLU
- **3 epochs per tuning run**

The search is a **small grid search** rather than an exhaustive search. The final MLP is retrained for a few more epochs using the best validation configuration.

For reproducibility and quick execution, the notebook uses a **stratified subset** if the full dataset is not already available. Replace the dataset path with the downloaded dataset folder/file.


In [ ]:

# Install only if required:
# !pip install numpy pandas matplotlib seaborn scikit-learn tensorflow pillow

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.utils import to_categorical

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)



## 1. Load and preprocess the dataset

Set `DATA_PATH` to the downloaded English Handwritten Characters Dataset.

The loader below supports either:
1. a CSV containing pixel columns plus a label column, or
2. an image-folder structure where each class is a subfolder.


In [ ]:

DATA_PATH = "English Handwritten Characters Dataset"   # <-- CHANGE THIS PATH
MAX_SAMPLES = 12000   # keeps execution manageable; set None to use all available samples

def load_dataset(path, max_samples=12000):
    if os.path.isfile(path) and path.lower().endswith(".csv"):
        df = pd.read_csv(path)
        # Common label names
        label_candidates = [c for c in ["label", "Label", "class", "Class", "character"] if c in df.columns]
        if not label_candidates:
            raise ValueError("CSV found, but no label column was detected.")
        label_col = label_candidates[0]
        y = df[label_col].values
        X = df.drop(columns=[label_col]).values.astype("float32")
        return X, y

    if not os.path.isdir(path):
        raise FileNotFoundError(
            f"Dataset path not found: {path}\n"
            "Download the dataset and update DATA_PATH."
        )

    # Image-folder loader
    from PIL import Image

    class_names = sorted([
        d for d in os.listdir(path)
        if os.path.isdir(os.path.join(path, d))
    ])
    if not class_names:
        raise ValueError("No class subfolders found.")

    X, y = [], []
    for label in class_names:
        folder = os.path.join(path, label)
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            try:
                img = Image.open(fpath).convert("L").resize((28, 28))
                X.append(np.asarray(img, dtype="float32").reshape(-1))
                y.append(label)
            except Exception:
                pass

    X, y = np.asarray(X), np.asarray(y)

    if max_samples is not None and len(X) > max_samples:
        # Stratified subset
        idx_train, idx_keep = train_test_split(
            np.arange(len(X)), train_size=max_samples,
            stratify=y, random_state=42
        )
        X, y = X[idx_train], y[idx_train]

    return X, y

X, y = load_dataset(DATA_PATH, MAX_SAMPLES)

X = X / 255.0
le = LabelEncoder()
y_int = le.fit_transform(y)
num_classes = len(le.classes_)
y_cat = to_categorical(y_int, num_classes)

X_train, X_temp, y_train, y_temp, yi_train, yi_temp = train_test_split(
    X, y_cat, y_int, test_size=0.30, stratify=y_int, random_state=42
)
X_val, X_test, y_val, y_test, yi_val, yi_test = train_test_split(
    X_temp, y_temp, yi_temp, test_size=0.50, stratify=yi_temp, random_state=42
)

print("Samples:", len(X))
print("Classes:", num_classes)
print("Train/Validation/Test:", len(X_train), len(X_val), len(X_test))
print("Input features:", X_train.shape[1])



## 2. Model A — Single-Layer Perceptron (PLA)

The PLA uses a step activation and the update:

\[
w_{t+1}=w_t+\eta(y-\hat y)x
\]

For this 62-class problem, a multiclass one-vs-rest implementation is used. Each class has one weight vector and bias. The class with the largest score is selected.


In [ ]:

class PLA:
    def __init__(self, n_features, n_classes, lr=0.01, epochs=8):
        self.W = np.zeros((n_classes, n_features), dtype=np.float32)
        self.b = np.zeros(n_classes, dtype=np.float32)
        self.lr = lr
        self.epochs = epochs
        self.errors_ = []

    def predict(self, X):
        scores = X @ self.W.T + self.b
        return np.argmax(scores, axis=1)

    def fit(self, X, y):
        for epoch in range(self.epochs):
            errors = 0
            for xi, yi in zip(X, y):
                pred = np.argmax(self.W @ xi + self.b)
                if pred != yi:
                    self.W[yi] += self.lr * xi
                    self.b[yi] += self.lr
                    self.W[pred] -= self.lr * xi
                    self.b[pred] -= self.lr
                    errors += 1
            self.errors_.append(errors)
        return self

pla = PLA(X_train.shape[1], num_classes, lr=0.01, epochs=8)
pla.fit(X_train, yi_train)
pla_pred = pla.predict(X_test)

pla_metrics = {
    "Accuracy": accuracy_score(yi_test, pla_pred),
    "Precision": precision_score(yi_test, pla_pred, average="macro", zero_division=0),
    "Recall": recall_score(yi_test, pla_pred, average="macro", zero_division=0),
    "F1": f1_score(yi_test, pla_pred, average="macro", zero_division=0)
}
print("PLA results:", pla_metrics)


In [ ]:

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(pla.errors_) + 1), pla.errors_, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Number of classification errors")
plt.title("PLA Convergence")
plt.grid(True)
plt.show()



## 3. Model B — MLP and lightweight hyperparameter tuning

The lab sheet asks for tuning of learning rate, batch size, optimizer and activation. To keep execution short, the main comparison below tunes **optimizer and learning rate** while keeping the architecture, activation and batch size fixed.

**Fixed values:** hidden layer = 64 neurons, activation = ReLU, batch size = 64.

**Search:** Adam/SGD × 0.001/0.01, for only 3 epochs each.

Validation accuracy is used to select the configuration. The test set is kept separate until final evaluation.


In [ ]:

def build_mlp(lr, optimizer_name, activation="relu", hidden_units=64):
    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(hidden_units, activation=activation),
        Dense(num_classes, activation="softmax")
    ])

    if optimizer_name == "adam":
        opt = Adam(learning_rate=lr)
    else:
        opt = SGD(learning_rate=lr)

    model.compile(
        optimizer=opt,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

search_space = [
    ("adam", 0.001),
    ("adam", 0.01),
    ("sgd", 0.001),
    ("sgd", 0.01)
]

tuning_results = []

for opt_name, lr in search_space:
    model = build_mlp(lr, opt_name)
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=3,
        batch_size=64,
        verbose=0
    )
    best_val_acc = max(hist.history["val_accuracy"])
    tuning_results.append({
        "optimizer": opt_name,
        "learning_rate": lr,
        "best_val_accuracy": best_val_acc
    })

tuning_df = pd.DataFrame(tuning_results).sort_values(
    "best_val_accuracy", ascending=False
).reset_index(drop=True)

display(tuning_df)



### Hyperparameter selection

The configuration with the highest validation accuracy is selected. This is a practical choice because validation data is used for model selection, while the test set remains unseen during tuning.


In [ ]:

best_opt = tuning_df.loc[0, "optimizer"]
best_lr = float(tuning_df.loc[0, "learning_rate"])

print("Selected optimizer:", best_opt)
print("Selected learning rate:", best_lr)
print("Selected hidden units: 64")
print("Selected activation: ReLU")
print("Selected batch size: 64")



## 4. Train the selected MLP and evaluate


In [ ]:

mlp = build_mlp(best_lr, best_opt, activation="relu", hidden_units=64)

history = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=64,
    verbose=1
)

test_prob = mlp.predict(X_test, verbose=0)
mlp_pred = np.argmax(test_prob, axis=1)

mlp_metrics = {
    "Accuracy": accuracy_score(yi_test, mlp_pred),
    "Precision": precision_score(yi_test, mlp_pred, average="macro", zero_division=0),
    "Recall": recall_score(yi_test, mlp_pred, average="macro", zero_division=0),
    "F1": f1_score(yi_test, mlp_pred, average="macro", zero_division=0)
}

print("MLP results:")
for k, v in mlp_metrics.items():
    print(f"{k}: {v:.4f}")


In [ ]:

comparison = pd.DataFrame([pla_metrics, mlp_metrics], index=["PLA", "MLP"])
display(comparison.round(4))


In [ ]:

# Training/validation convergence
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MLP Accuracy Convergence")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("MLP Loss Convergence")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# Confusion matrix
cm = confusion_matrix(yi_test, mlp_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues", cbar=False)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("MLP Confusion Matrix")
plt.show()



## 5. ROC curve — micro average

For the multiclass problem, a micro-average ROC curve is plotted by flattening the one-hot test labels and predicted probabilities.


In [ ]:

y_test_bin = y_test.ravel()
y_score = test_prob.ravel()

fpr, tpr, _ = roc_curve(y_test_bin, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Micro-average ROC (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("MLP ROC Curve")
plt.legend()
plt.grid(True)
plt.show()



## 6. Observation / Analysis

### Justification of chosen hyperparameters

A small systematic grid search was used because the experiment requires hyperparameter tuning, but repeated training on a 62-class image dataset can be computationally expensive. Four configurations were compared using validation accuracy: Adam with learning rates 0.001 and 0.01, and SGD with learning rates 0.001 and 0.01. The configuration with the highest validation accuracy was selected. Adam was considered useful when it reached higher validation accuracy or converged faster because its adaptive learning-rate mechanism adjusts parameter updates using estimates of first and second moments of gradients. The learning rate was kept small enough to avoid unstable updates while still allowing useful progress within a few epochs.

ReLU was selected for the hidden layer because the MLP must learn nonlinear decision boundaries, while the output layer uses softmax because the task has multiple character classes. Categorical cross-entropy was used because the target is a multiclass classification problem. A single hidden layer with 64 neurons was deliberately used as a compact architecture: it provides nonlinear representation capacity without adding unnecessary training time. Batch size 64 gives a practical balance between stable gradient estimates and computational cost.

### PLA vs MLP

PLA uses a linear decision function with a step activation and therefore has limited ability to separate classes that require nonlinear decision boundaries. The MLP contains a hidden layer with a nonlinear activation and learns parameters through backpropagation, allowing it to model more complex class boundaries. Therefore, the MLP is expected to perform better on handwritten character images.

### Effect of tuning

The tuning table shows which optimizer and learning rate produced the best validation accuracy in this run. Optimizer choice can affect convergence speed and final validation performance. A learning rate that is too small may learn slowly, while a larger value can produce faster progress but may make optimization less stable.

### More hidden layers

Adding hidden layers does not automatically improve performance. Extra layers increase model capacity and computation. If the dataset and training procedure do not require the additional capacity, the extra parameters may provide little improvement or may increase overfitting.

### Overfitting

Compare training and validation curves. If training accuracy continues increasing while validation accuracy stops improving or validation loss increases, this indicates overfitting. Possible mitigation methods include early stopping, regularization, dropout, data augmentation and reducing model complexity.



## Final report values

After running the notebook, copy the numerical values from the `comparison` and `tuning_df` tables into the observation section of the record.

The required final report components from the lab sheet are:
- Aim and Objective
- Preprocessing Steps
- PLA Implementation and Results
- MLP Implementation and Results
- Justification for Chosen Hyperparameters
- PLA vs MLP comparison
- Confusion Matrix and ROC Curve
- Observations and Analysis
